## 1 - Model 3 Random Forest

Based on evidence from recent studies, tree-based ensemble models such as  Random Forest seem the most suitable choice for intrusion detection using the NSL-KDD dataset. RF consistently achieved accuracy above 99% across multiple experiments (Abrar et al., 2020; Hegde et al., 2024; Hong et al., 2021; Gawand & Meesala, 2025). It effectively handles the dataset’s mixed-type features and class imbalance, while providing high interpretability and resilience to overfitting. Compared to deep learning models, RF offers an optimal trade-off between performance, computational efficiency and ease of implementation. This makes it a very appropriate algorithm for both academic experimentation and practical network intrusion detection.

Main goal is to:
* Identify the most informative NSL-KDD features
* Improve multi-class prediction performance over logistic regression
* Build a more expressive model without overfitting
* Evaluate Model 3, generate interpretable insights to produce a final, optimized RF model ready for test evaluation

Furthermore, in Model 3, the RF algorithm is intentionally used twice, but for two different purposes. First, an initial RF is trained on the full feature set to compute feature importances. This model is not used for prediction, it is only used to rank the NSL-KDD attributes and identify which features carry meaningful signal versus which ones introduce noise or redundancy. Based on this ranking, a reduced subset of the most informative features is selected. In the second stage, a new RF model is traine and only on the selected features, to serve as the actual predictive classifier. This separation ensures that the final model benefits from a cleaner and lower-dimensional feature space, and enables an assessment of how feature selection influences model performance.



In [1]:
# Start by importing all required libs...
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from pathlib import Path
from utils import load_split, FEATURE_COLUMNS



In [ ]:
# Part one: selecting the most importan features
DATA_DIR   = Path("data/processed")
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

X_train, y_train = load_split("train", DATA_DIR)
X_val,   y_val   = load_split("val", DATA_DIR)
X_test,  y_test  = load_split("test", DATA_DIR)

print("The shapes:")
print("  X_train:", X_train.shape)
print("  X_val:  ", X_val.shape)
print("  X_test: ", X_test.shape)

assert X_train.shape[1] == len(FEATURE_COLUMNS), "Feature dimension issue..."


# RF used only for ranking features
rf_feat_imp = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample", # minority classes are not ignored
)
rf_feat_imp.fit(X_train, y_train)

importances = rf_feat_imp.feature_importances_

feat_imp_df = (
    pd.DataFrame({"feature": FEATURE_COLUMNS, "importance": importances})
    .sort_values("importance", ascending=False)
)

print("\nTop 10 features by RF importance:")
print(feat_imp_df.head(10))

TOP_F = 20 # 50% dimensionality reduction
top_features = feat_imp_df.head(TOP_F)["feature"].tolist()
print(f"\nSelected top-{TOP_F} features:\n", top_features)

# create index mapping
name_to_idx = {name: idx for idx, name in enumerate(FEATURE_COLUMNS)}
selected_idx = [name_to_idx[f] for f in top_features]

X_train_sel = X_train[:, selected_idx]
X_val_sel   = X_val[:,   selected_idx]
X_test_sel  = X_test[:,  selected_idx]

print("\n The Shapes after feature selection:")
print("  X_train_sel:", X_train_sel.shape)
print("  X_val_sel:  ", X_val_sel.shape)
print("  X_test_sel: ", X_test_sel.shape)

The shapes:
  X_train: (100778, 41)
  X_val:   (25195, 41)
  X_test:  (22544, 41)

Top 10 features by RF importance:
                        feature  importance
4                     src_bytes    0.097232
5                     dst_bytes    0.064228
2                       service    0.052866
32           dst_host_srv_count    0.048483
31               dst_host_count    0.047082
7                wrong_fragment    0.046191
36  dst_host_srv_diff_host_rate    0.043701
35  dst_host_same_src_port_rate    0.037076
23                    srv_count    0.033839
0                      duration    0.032905

Selected top-20 features:
 ['src_bytes', 'dst_bytes', 'service', 'dst_host_srv_count', 'dst_host_count', 'wrong_fragment', 'dst_host_srv_diff_host_rate', 'dst_host_same_src_port_rate', 'srv_count', 'duration', 'dst_host_diff_srv_rate', 'dst_host_same_srv_rate', 'count', 'protocol_type', 'dst_host_srv_serror_rate', 'dst_host_serror_rate', 'hot', 'land', 'flag', 'logged_in']

Shapes after feature 